# LangGraph x A2A x MCP — complex problem-solving lab

**Goal:** run a full multi-agent, multi-protocol stack end to end with today's mainstream
orchestration pieces, see *all* the agentic code you must write by hand, and catalog the
real-world problems that show up. This is the "before graxella" baseline.

## Architecture

```mermaid
flowchart LR
    U[Incident report] --> S[LangGraph supervisor\nqwen2.5:7b via Ollama]
    S -- "A2A JSON-RPC :9101" --> D[diagnostics-agent\nLangGraph ReAct]
    S -- "A2A JSON-RPC :9102" --> R[remediation-agent\nLangGraph ReAct]
    D -- "MCP streamable-http :8901" --> M[(ops-tools MCP server\nmetrics / deps / deploys)]
    R -- "MCP streamable-http :8901" --> M2[(ops-tools MCP server\nrunbooks / tickets)]
```

Four OS processes: 1 MCP tool server, 2 A2A agent servers, 1 notebook kernel (orchestrator).
Every LLM call is a **local Ollama** model — no cloud keys needed.

## Reference projects this is modeled on
- [a2aproject/A2A](https://github.com/a2aproject/A2A) + [a2aproject/a2a-samples](https://github.com/a2aproject/a2a-samples) — A2A protocol & the LangGraph sample agent
- [langchain-ai/langgraph](https://github.com/langchain-ai/langgraph) — supervisor / ReAct graphs
- [langchain-ai/langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters) — MCP tools as LangChain tools
- [modelcontextprotocol/python-sdk](https://github.com/modelcontextprotocol/python-sdk) — FastMCP server

> ⚠️ **First real-world problem, before any code runs:** the installed `a2a-sdk 1.1.0` is the new
> **protobuf-based** API. Nearly every GitHub sample (including a2a-samples) targets the old 0.x
> pydantic API (`TextPart`, `A2AStarletteApplication`, `new_agent_text_message`) — none of those
> names exist any more. Everything below was written against the installed package, not the samples.


## The mission (a genuinely multi-hop problem)

> **INCIDENT:** since 09:40 UTC `checkout-api` p99 latency is 4.2s (baseline 0.5s), error rate 12%.
> Customers cannot complete payments. Find the root cause and produce a remediation plan + ticket.

**Hidden ground truth** (baked into the MCP server's synthetic world):
`payment-gateway v2.14.1` (deployed 09:35 UTC, "async DB connection pool refactor") leaks
connections → pool saturates (200/200 in use, wait queue 341) → checkout-api 502/504s.
There is a **red herring**: `redis-cache` memory is at 97% — but it has been for 3 weeks and its
hit rate is stable. Solving this correctly requires: metrics → dependency walk → deploy history →
ruling out the herring. A single tool call cannot answer it.


In [1]:
# 0. Environment check --------------------------------------------------------
# Requires (in THIS kernel's environment):
#   pip install a2a-sdk langgraph langchain-ollama mcp langchain-mcp-adapters fastapi uvicorn httpx
import sys, subprocess, time, importlib.metadata as im

for pkg in ["a2a-sdk", "langgraph", "langchain-ollama", "mcp", "langchain-mcp-adapters"]:
    try:
        print(f"{pkg:24s} {im.version(pkg)}")
    except im.PackageNotFoundError:
        print(f"{pkg:24s} MISSING -> pip install {pkg}")

import httpx
MODEL = "qwen2.5:7b"           # every agent below uses this; try "qwen3-coder:latest" for quality
tags = httpx.get("http://localhost:11434/api/tags", timeout=5).json()
names = [m["name"] for m in tags["models"]]
print("\nOllama models:", names)
assert MODEL in names, f"ollama pull {MODEL} first"
print("interpreter:", sys.executable)

a2a-sdk                  1.1.0
langgraph                1.1.10
langchain-ollama         1.1.0
mcp                      1.29.0
langchain-mcp-adapters   0.3.2



Ollama models: ['qwen3-coder:latest', 'qwen2.5:0.5b', 'qwen2.5:7b', 'qwen2.5:3b', 'qwen2.5:latest', 'nomic-embed-text:latest', 'deepseek-r1:latest', 'embeddinggemma:latest', 'deepseek-r1:14b', 'llama3.2:latest', 'llama3:latest', 'mistral:latest']
interpreter: C:\Python313\python.exe


---
## Layer 1 — MCP tool server

Five ops tools over **streamable-http** (the transport real deployments use, and the one that
works reliably from a Windows notebook — stdio MCP servers fight with Jupyter's event loop).
The synthetic world with the ground truth lives here.


In [2]:
%%writefile lab_a2a_mcp/mcp_ops_server.py
"""MCP tool server for the incident-response lab.

Exposes five ops tools over streamable-http (the transport real deployments use).
Run:  python mcp_ops_server.py   ->  http://127.0.0.1:8901/mcp
"""
import json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ops-tools", host="127.0.0.1", port=8901)

# ---- synthetic incident world -------------------------------------------------
# Ground truth: payment-gateway v2.14.1 (deployed 09:35 UTC) leaks DB connections
# -> pool exhaustion -> checkout-api latency/error spike. redis is a red herring.

METRICS = {
    "checkout-api": {
        "p99_latency_ms": 4180, "baseline_p99_ms": 510, "error_rate_pct": 12.4,
        "note": "errors began 09:41 UTC, all 502/504 from upstream calls",
    },
    "payment-gateway": {
        "p99_latency_ms": 3920, "baseline_p99_ms": 300, "error_rate_pct": 9.8,
        "db_connection_pool": {"in_use": 200, "max": 200, "wait_queue": 341},
        "note": "pool saturated since 09:39 UTC; connections not being released",
    },
    "inventory-svc": {"p99_latency_ms": 88, "baseline_p99_ms": 85, "error_rate_pct": 0.1},
    "redis-cache": {
        "p99_latency_ms": 4, "baseline_p99_ms": 3, "memory_used_pct": 97,
        "evictions_per_min": 220, "note": "memory high for 3 weeks; hit rate stable at 94%",
    },
    "bank-connector": {"p99_latency_ms": 240, "baseline_p99_ms": 235, "error_rate_pct": 0.2},
}

DEPLOYS = {
    "payment-gateway": [
        {"version": "v2.14.1", "at": "09:35 UTC today", "change": "refactor: async DB connection pool handling"},
        {"version": "v2.14.0", "at": "6 days ago", "change": "add settlement retries"},
    ],
    "checkout-api": [{"version": "v8.2.0", "at": "3 days ago", "change": "copy changes on receipt page"}],
    "inventory-svc": [{"version": "v3.1.9", "at": "12 days ago", "change": "index rebuild job"}],
    "redis-cache": [], "bank-connector": [],
}

DEPENDENCIES = {
    "checkout-api": ["payment-gateway", "inventory-svc", "redis-cache"],
    "payment-gateway": ["redis-cache", "bank-connector"],
    "inventory-svc": ["redis-cache"],
}

RUNBOOKS = [
    {"id": "RB-101", "title": "Database connection pool exhaustion",
     "steps": ["confirm pool in_use == max and wait_queue growing",
               "identify the release that changed pool/connection handling",
               "roll back that release (see RB-033)",
               "if rollback impossible: bump pool max 2x as a stopgap and recycle pods"]},
    {"id": "RB-207", "title": "Redis memory pressure",
     "steps": ["check eviction rate and hit rate", "if hit rate stable, schedule capacity work - NOT an incident page",
               "if hit rate collapsing, scale replica and warm cache"]},
    {"id": "RB-033", "title": "Standard rollback procedure",
     "steps": ["freeze deploys for the service", "helm rollback <service> to previous revision",
               "watch golden signals for 15 min", "file post-incident ticket with root cause"]},
]

TICKETS: list[dict] = []

# ---- tools --------------------------------------------------------------------

@mcp.tool()
def get_service_metrics(service: str) -> str:
    """Current golden-signal metrics for a service (latency, errors, saturation)."""
    m = METRICS.get(service)
    return json.dumps(m, indent=1) if m else f"unknown service '{service}'. known: {sorted(METRICS)}"

@mcp.tool()
def get_dependency_map(service: str) -> str:
    """Downstream dependencies of a service (what it calls)."""
    return json.dumps({"service": service, "calls": DEPENDENCIES.get(service, [])})

@mcp.tool()
def get_recent_deploys(service: str) -> str:
    """Recent deploys for a service, newest first."""
    return json.dumps(DEPLOYS.get(service, []), indent=1)

@mcp.tool()
def search_runbooks(query: str) -> str:
    """Keyword search over incident runbooks; returns matching runbooks with steps."""
    q = query.lower()
    hits = [rb for rb in RUNBOOKS if any(w in (rb["title"] + " " + " ".join(rb["steps"])).lower()
                                         for w in q.split())]
    return json.dumps(hits or RUNBOOKS, indent=1)

@mcp.tool()
def create_ticket(title: str, severity: str, body: str) -> str:
    """File an incident ticket. severity: sev1|sev2|sev3."""
    t = {"id": f"OPS-{1000 + len(TICKETS)}", "title": title, "severity": severity, "body": body}
    TICKETS.append(t)
    return json.dumps(t)

if __name__ == "__main__":
    mcp.run(transport="streamable-http")


Overwriting lab_a2a_mcp/mcp_ops_server.py


In [3]:
# Launch the MCP server as its own process and wait until it answers ----------
import pathlib, subprocess, sys, time, httpx

LAB = pathlib.Path("lab_a2a_mcp").resolve()
PROCS = {}

def launch(script: str, port: int, probe: str) -> None:
    PROCS[script] = subprocess.Popen([sys.executable, script], cwd=LAB)
    for _ in range(60):
        try:
            httpx.get(f"http://127.0.0.1:{port}{probe}", timeout=2)
            print(f"{script} up on :{port} (pid {PROCS[script].pid})")
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"{script} did not come up — check the process output")

# FastMCP answers on /mcp (405 on GET is fine — it's alive)
launch("mcp_ops_server.py", 8901, "/mcp")

mcp_ops_server.py up on :8901 (pid 37264)


In [4]:
# Discover the MCP tools exactly the way any LangChain/LangGraph agent would --
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({"ops": {"transport": "streamable_http",
                                           "url": "http://127.0.0.1:8901/mcp"}})
mcp_tools = await mcp_client.get_tools()
for t in mcp_tools:
    print(f"- {t.name}: {t.description}")

# one direct call to prove the wire works
out = await next(t for t in mcp_tools if t.name == "get_service_metrics").ainvoke(
    {"service": "payment-gateway"})
print("\nget_service_metrics('payment-gateway') ->")
print(out)

C:\Users\Sridhar\AppData\Roaming\Python\Python313\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


- get_service_metrics: Current golden-signal metrics for a service (latency, errors, saturation).
- get_dependency_map: Downstream dependencies of a service (what it calls).
- get_recent_deploys: Recent deploys for a service, newest first.
- search_runbooks: Keyword search over incident runbooks; returns matching runbooks with steps.
- create_ticket: File an incident ticket. severity: sev1|sev2|sev3.



get_service_metrics('payment-gateway') ->
[{'type': 'text', 'text': '{\n "p99_latency_ms": 3920,\n "baseline_p99_ms": 300,\n "error_rate_pct": 9.8,\n "db_connection_pool": {\n  "in_use": 200,\n  "max": 200,\n  "wait_queue": 341\n },\n "note": "pool saturated since 09:39 UTC; connections not being released"\n}', 'id': 'lc_503ce946-1d73-4997-8b15-63e4993ef713'}]


---
## Layer 2 — two specialist agents behind the A2A protocol

Each specialist is a **LangGraph ReAct agent + MCP tools**, exposed as an **A2A server**.
`a2a_common.py` is every line of glue you must hand-write today: an `AgentExecutor` bridging
A2A protobuf ⇄ LangChain messages, the agent card, and mounting JSON-RPC + REST + card routes
on FastAPI. Note there is **no framework help** for any of this — this file *is* the integration.


In [5]:
%%writefile lab_a2a_mcp/a2a_common.py
"""Boilerplate needed to expose one LangGraph agent over the A2A protocol.

Every line here is glue a developer must hand-write today: wrap the graph in an
AgentExecutor, translate LangChain messages <-> A2A protobuf, build the agent
card, mount JSON-RPC + REST + card routes on FastAPI, run uvicorn.
"""
import uvicorn
from fastapi import FastAPI

from a2a.helpers import new_text_message
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import (
    add_a2a_routes_to_fastapi,
    create_agent_card_routes,
    create_jsonrpc_routes,
    create_rest_routes,
)
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentInterface, AgentSkill


class LangGraphAgentExecutor(AgentExecutor):
    """Bridges an A2A request into a LangGraph agent and back."""

    def __init__(self, graph, name: str):
        self.graph = graph
        self.name = name

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.get_user_input()  # A2A protobuf Message -> plain text
        print(f"[{self.name}] A2A task in: {task[:120]!r}")
        result = await self.graph.ainvoke(
            {"messages": [{"role": "user", "content": task}]},
            {"recursion_limit": 40},
        )
        reply = result["messages"][-1].content  # LangChain message -> text
        n_tool_calls = sum(len(getattr(m, "tool_calls", []) or []) for m in result["messages"])
        print(f"[{self.name}] done: {len(result['messages'])} messages, {n_tool_calls} tool calls")
        # text -> A2A protobuf Message (role defaults to ROLE_AGENT)
        await event_queue.enqueue_event(new_text_message(reply, context_id=context.context_id))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError


def build_agent_card(name: str, description: str, url: str, skills: list[AgentSkill]) -> AgentCard:
    return AgentCard(
        name=name,
        description=description,
        version="1.0.0",
        supported_interfaces=[AgentInterface(url=url, protocol_binding="JSONRPC")],
        capabilities=AgentCapabilities(streaming=True),
        default_input_modes=["text/plain"],
        default_output_modes=["text/plain"],
        skills=skills,
    )


def serve(graph, card: AgentCard, port: int) -> None:
    handler = DefaultRequestHandler(
        agent_executor=LangGraphAgentExecutor(graph, card.name),
        task_store=InMemoryTaskStore(),
        agent_card=card,
    )
    app = FastAPI()
    add_a2a_routes_to_fastapi(
        app,
        agent_card_routes=create_agent_card_routes(card),
        jsonrpc_routes=create_jsonrpc_routes(handler, rpc_url="/"),
        rest_routes=create_rest_routes(handler),
    )
    uvicorn.run(app, host="127.0.0.1", port=port, log_level="warning")


Overwriting lab_a2a_mcp/a2a_common.py


In [6]:
%%writefile lab_a2a_mcp/a2a_diagnostics.py
"""Diagnostics specialist: a LangGraph ReAct agent with read-only MCP ops tools,
served over A2A on port 9101."""
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

from a2a.types import AgentSkill
from a2a_common import build_agent_card, serve

MODEL = "qwen2.5:7b"
MCP_URL = "http://127.0.0.1:8901/mcp"
PORT = 9101

PROMPT = """You are a production diagnostics specialist.
Given an incident description, find the ROOT CAUSE, not just symptoms:
1. Pull metrics for the affected service, then walk its dependency map and pull
   metrics for each dependency that looks implicated.
2. Check recent deploys for any service whose metrics look abnormal.
3. Distinguish real causes from red herrings (a metric can be bad but stable/unrelated).
Answer with: root-cause service, the offending change if any, the causal chain,
and what you ruled out. Be concrete and cite the numbers you saw."""


async def build_graph():
    client = MultiServerMCPClient({"ops": {"transport": "streamable_http", "url": MCP_URL}})
    tools = await client.get_tools()
    wanted = {"get_service_metrics", "get_dependency_map", "get_recent_deploys"}
    tools = [t for t in tools if t.name in wanted]
    llm = ChatOllama(model=MODEL, temperature=0)
    return create_react_agent(llm, tools, prompt=PROMPT)


if __name__ == "__main__":
    graph = asyncio.run(build_graph())
    card = build_agent_card(
        name="diagnostics-agent",
        description="Finds the root cause of production incidents using live metrics, dependency maps and deploy history.",
        url=f"http://127.0.0.1:{PORT}",
        skills=[AgentSkill(
            id="root_cause_analysis", name="Root cause analysis",
            description="Traces an incident through service dependencies to the causing change.",
            tags=["diagnostics", "observability"],
            examples=["Why is checkout-api slow since 09:40 UTC?"],
        )],
    )
    print(f"diagnostics-agent up on :{PORT} with tools: metrics, deps, deploys")
    serve(graph, card, PORT)


Overwriting lab_a2a_mcp/a2a_diagnostics.py


In [7]:
%%writefile lab_a2a_mcp/a2a_remediation.py
"""Remediation specialist: a LangGraph ReAct agent with runbook + ticket MCP tools,
served over A2A on port 9102."""
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

from a2a.types import AgentSkill
from a2a_common import build_agent_card, serve

MODEL = "qwen2.5:7b"
MCP_URL = "http://127.0.0.1:8901/mcp"
PORT = 9102

PROMPT = """You are an incident remediation planner.
You receive a diagnosed root cause. Your job:
1. search_runbooks for the failure mode and pick the applicable runbook(s).
2. Produce a concrete, ordered remediation plan (name the exact service/version).
3. create_ticket exactly once: severity sev1 if customers are impacted, title
   naming the root cause, body containing your full plan.
Answer with the plan and the ticket id you created."""


async def build_graph():
    client = MultiServerMCPClient({"ops": {"transport": "streamable_http", "url": MCP_URL}})
    tools = await client.get_tools()
    wanted = {"search_runbooks", "create_ticket", "get_recent_deploys"}
    tools = [t for t in tools if t.name in wanted]
    llm = ChatOllama(model=MODEL, temperature=0)
    return create_react_agent(llm, tools, prompt=PROMPT)


if __name__ == "__main__":
    graph = asyncio.run(build_graph())
    card = build_agent_card(
        name="remediation-agent",
        description="Turns a diagnosed root cause into an ordered remediation plan grounded in runbooks, and files the incident ticket.",
        url=f"http://127.0.0.1:{PORT}",
        skills=[AgentSkill(
            id="remediation_planning", name="Remediation planning",
            description="Runbook-grounded fix plan plus incident ticket.",
            tags=["remediation", "runbooks", "ticketing"],
            examples=["Root cause: payment-gateway v2.14.1 leaks DB connections. Plan the fix."],
        )],
    )
    print(f"remediation-agent up on :{PORT} with tools: runbooks, tickets, deploys")
    serve(graph, card, PORT)


Overwriting lab_a2a_mcp/a2a_remediation.py


In [8]:
# Launch both agents and verify their A2A agent cards -------------------------
launch("a2a_diagnostics.py", 9101, "/.well-known/agent-card.json")
launch("a2a_remediation.py", 9102, "/.well-known/agent-card.json")

import json
for port in (9101, 9102):
    card = httpx.get(f"http://127.0.0.1:{port}/.well-known/agent-card.json", timeout=5).json()
    print(f"\n:{port}  {card['name']}  (A2A protocol {card.get('protocolVersion')})")
    print("  ", card["description"])
    for s in card["skills"]:
        print(f"   skill: {s['id']} — {s['description']}")

a2a_diagnostics.py up on :9101 (pid 31760)


a2a_remediation.py up on :9102 (pid 23120)



:9101  diagnostics-agent  (A2A protocol 0.3)
   Finds the root cause of production incidents using live metrics, dependency maps and deploy history.
   skill: root_cause_analysis — Traces an incident through service dependencies to the causing change.



:9102  remediation-agent  (A2A protocol 0.3)
   Turns a diagnosed root cause into an ordered remediation plan grounded in runbooks, and files the incident ticket.
   skill: remediation_planning — Runbook-grounded fix plan plus incident ticket.


---
## Layer 3 — the LangGraph orchestrator (incident commander)

The supervisor has **no tools of its own** — each remote A2A agent is wrapped as a LangChain
tool doing a full A2A round trip. Two things below were only discovered by hitting them:

1. **`create_client` needs an injected `httpx.AsyncClient` with a long timeout.** The default
   times out before a local-LLM agent can even start answering (`A2AClientTimeoutError`).
2. The stream's oneof is `WhichOneof("payload")` — undocumented; found by reading the proto.


In [9]:
# A2A client plumbing + the two "remote agent as a tool" wrappers -------------
import httpx, time
from langchain_core.tools import tool
from a2a.client import ClientConfig, create_client
from a2a.helpers import get_stream_response_text, new_text_message
from a2a.types import Role, SendMessageRequest

DIAG_URL, REMED_URL = "http://127.0.0.1:9101", "http://127.0.0.1:9102"
_clients, HOP_LOG = {}, []

async def a2a_ask(url: str, text: str) -> str:
    """One A2A round trip; blocks until the remote agent finishes thinking."""
    if url not in _clients:
        cfg = ClientConfig(httpx_client=httpx.AsyncClient(timeout=httpx.Timeout(600.0)))
        _clients[url] = await create_client(url, cfg)   # resolves the agent card
    req = SendMessageRequest(message=new_text_message(text, role=Role.ROLE_USER))
    t0 = time.time()
    chunks = [get_stream_response_text(r) async for r in _clients[url].send_message(req)]
    reply = "\n".join(c for c in chunks if c) or "(remote agent returned no text)"
    HOP_LOG.append({"to": url, "sent_chars": len(text), "recv_chars": len(reply),
                    "secs": round(time.time() - t0, 1)})
    return reply

@tool
async def consult_diagnostics(incident_description: str) -> str:
    """Ask the remote diagnostics agent to find the root cause of an incident.
    Pass the full incident description; it has live metrics, dependency maps and deploy history."""
    print(f">>> A2A -> diagnostics-agent ({len(incident_description)} chars)")
    out = await a2a_ask(DIAG_URL, incident_description)
    print(f"<<< diagnostics-agent replied ({len(out)} chars)")
    return out

@tool
async def consult_remediation(diagnosed_root_cause: str) -> str:
    """Ask the remote remediation agent for a runbook-grounded fix plan + incident ticket.
    Pass the diagnosed root cause (service, version, causal chain)."""
    print(f">>> A2A -> remediation-agent ({len(diagnosed_root_cause)} chars)")
    out = await a2a_ask(REMED_URL, diagnosed_root_cause)
    print(f"<<< remediation-agent replied ({len(out)} chars)")
    return out

print("A2A tool wrappers ready")

A2A tool wrappers ready


In [10]:
# The supervisor graph ---------------------------------------------------------
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent   # deprecated alias; new home: langchain.agents.create_agent

SUPERVISOR_PROMPT = """You are the incident commander. You do NOT have direct access
to metrics or runbooks - you must delegate:
1. First call consult_diagnostics with the incident description to get the root cause.
2. Then call consult_remediation, passing the diagnosed root cause verbatim.
3. Finally write the incident summary: root cause, evidence, plan, ticket id.
Call each tool at most twice. Do not invent facts the specialists did not report."""

supervisor = create_react_agent(
    ChatOllama(model=MODEL, temperature=0),
    [consult_diagnostics, consult_remediation],
    prompt=SUPERVISOR_PROMPT,
)
print("supervisor nodes:", list(supervisor.get_graph().nodes))

supervisor nodes: ['__start__', 'agent', 'tools', '__end__']


C:\Users\Sridhar\AppData\Local\Temp\ipykernel_28712\3550462633.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  supervisor = create_react_agent(


In [11]:
# RUN THE MISSION ---------------------------------------------------------------
INCIDENT = """INCIDENT: since 09:40 UTC checkout-api p99 latency is 4.2s (baseline 0.5s)
and error rate is 12%. Customers cannot complete payments. Find the root cause and
produce a remediation plan with an incident ticket."""

t0 = time.time()
result = await supervisor.ainvoke(
    {"messages": [{"role": "user", "content": INCIDENT}]},
    {"recursion_limit": 20},
)
print(f"\n=== finished in {time.time()-t0:.0f}s, {len(result['messages'])} supervisor messages ===")
print("\n=== FINAL ANSWER ===\n")
print(result["messages"][-1].content)

>>> A2A -> diagnostics-agent (123 chars)


<<< diagnostics-agent replied (1276 chars)


>>> A2A -> remediation-agent (59 chars)


<<< remediation-agent replied (753 chars)



=== finished in 39s, 8 supervisor messages ===

=== FINAL ANSWER ===

Based on the diagnosis and the remediation plan, here is the incident summary:

- **Root Cause:** The recent change "copy changes on receipt page" in `checkout-api` version v8.2.0 led to increased latency and error rates, particularly due to issues with its dependencies, likely causing database connection pool exhaustion.
- **Evidence:** The p99 latency increased from 0.5s to 4.2s, and the error rate spiked to 12.4%. The dependency map showed issues with `checkout-api`'s upstream services, and the recent deploy history pointed to the "copy changes on receipt page" change.
- **Plan:** The ticket created for this incident is OPS-1000. The plan involves confirming if the database connection pool is exhausted, identifying the release that changed pool/connection handling, and either rolling back the release or bumping the pool max by 2x as a stopgap measure and recycling the pods.
- **Ticket ID:** OPS-1000

The incident

In [12]:
# Full transcript + hop economics — this is where the bodies are buried ---------
print("SUPERVISOR TRANSCRIPT")
for m in result["messages"]:
    tc = getattr(m, "tool_calls", None)
    label = m.type + (f" -> {[t['name'] for t in tc]}" if tc else "")
    print(f"  [{label:32s}] {str(m.content)[:120]!r}")

print("\nA2A HOPS (chars shipped as opaque text)")
for h in HOP_LOG:
    print(f"  -> {h['to']}  sent={h['sent_chars']:5d}  recv={h['recv_chars']:5d}  {h['secs']}s")

print("\nGround truth was payment-gateway v2.14.1 — check the ticket above:")
print("  did the plan roll back the RIGHT service? In our runs it often did not.")

SUPERVISOR TRANSCRIPT
  [human                           ] 'INCIDENT: since 09:40 UTC checkout-api p99 latency is 4.2s (baseline 0.5s)\nand error rate is 12%. Customers cannot compl'
  [ai -> ['consult_diagnostics']   ] ''
  [tool                            ] "Error invoking tool 'consult_diagnostics' with kwargs {'incident_description': {'type': 'string', 'value': 'since 09:40 "
  [ai -> ['consult_diagnostics']   ] 'It seems there was an issue with the input format. Let me try again with the correct format.\n\n'
  [tool                            ] 'Based on the provided metrics and information:\n\n1. **Root Cause Service**: `checkout-api`\n2. **Offending Change**: The c'
  [ai -> ['consult_remediation']   ] ''
  [tool                            ] '-ticket-id: OPS-1000\n\nThe ticket has been created with the following details:\n\n- **Title:** Database connection pool exh'
  [ai                              ] 'Based on the diagnosis and the remediation plan, here is the incident summar

---
## What actually happened — problems observed with this stack

Everything below occurred **in real runs of this exact notebook** (your rerun will vary —
that variance is itself finding #0: none of this is reproducible).

| # | Problem | Layer | What we saw |
|---|---------|-------|-------------|
| 1 | **Sample rot / API churn** | A2A, LangGraph | `a2a-sdk 1.1.0` is protobuf; all GitHub samples use the dead 0.x pydantic API. `create_react_agent` deprecated mid-1.x. Code written 6 months ago does not import. |
| 2 | **Defaults hostile to agentic latency** | A2A | Default httpx timeout kills any remote agent that thinks >5s → `A2AClientTimeoutError`. Fix (inject your own `AsyncClient`) is undocumented. |
| 3 | **Malformed tool calls become silent in-context retries** | LangGraph | qwen2.5:7b's first call passed `{"type":"string","value":...}` instead of a string. LangGraph stuffed the error string into context and the model retried. Invisible unless you dump the transcript; burns tokens and can loop. |
| 4 | **No evidence gate — confident wrong answers cascade** | all | Diagnostics blamed `checkout-api v8.2.0` ("copy changes on receipt page"!) for an 8× latency spike. Ground truth was `payment-gateway v2.14.1`'s pool leak. Nothing challenged it; remediation filed a sev1 ticket to **roll back the wrong service**. |
| 5 | **Telephone-game handoffs** | A2A | The supervisor compressed a 1,452-char diagnosis into a ~50-char string for remediation. A2A ships opaque text Parts — no typed handoff envelope, no required evidence fields, no verbatim guarantee. |
| 6 | **Red-herring susceptibility** | model+tools | Asked casually about redis, the 7b model declared 3-week-stable 97% memory "the root cause". Raw metrics without priors invite this. |
| 7 | **Three incompatible envelopes, zero shared trace** | all | LangChain messages ⇄ A2A protobuf ⇄ MCP JSON-RPC, each boundary hand-glued (`a2a_common.py` exists only for this). No trace id crosses a boundary: debugging = correlating 3 process logs by timestamp. |
| 8 | **Ops burden for a toy** | infra | 4 processes, hardcoded ports, no health/restart/discovery story. `.well-known/agent-card.json` is discovery-by-URL — you must already know where everyone lives. |
| 9 | **No memory, no outcome learning** | all | Rerun the notebook: same wrong diagnosis is possible again. Nothing records that ticket OPS-1000 targeted the wrong service. The system cannot get better. |

### Why this matters (the graxella thesis, in one paragraph)
Every failure above is *between* the boxes, not inside them: LangGraph, A2A and MCP each work
as advertised in isolation. What's missing is the connective tissue — a typed handoff envelope
instead of opaque text (#5), an evidence gate that challenges a diagnosis before it becomes a
sev1 rollback ticket (#4, #6), one trace id across all three protocols (#7), and decision memory
so run N+1 knows what run N got wrong (#9). That connective tissue is exactly what graxella's
mesh + gate + mnema layers are for.

### Experiments to try
- Set `MODEL = "qwen3-coder:latest"` (30B): does #3/#4 improve, and at what latency cost?
- Weaken the supervisor prompt (drop "verbatim") and watch #5 get worse.
- Kill the MCP server mid-run: observe how the failure surfaces (or doesn't) three layers up.
- Ask both specialists the same question and diff the answers — no arbitration exists anywhere.


In [13]:
# Cleanup: stop all three server processes --------------------------------------
for name, p in PROCS.items():
    p.terminate()
    print(f"terminated {name} (pid {p.pid})")
PROCS.clear()

terminated mcp_ops_server.py (pid 37264)
terminated a2a_diagnostics.py (pid 31760)
terminated a2a_remediation.py (pid 23120)
